# Qwen3-TTS 1.7B — Colab TTS Test

Direct local inference for Japanese TTS using `Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice`.

**Recommended runtime:** A100 40/80GB, L4, or any CUDA GPU with enough VRAM.

This notebook measures:
- model load time
- generation latency
- audio duration
- real-time factor (RTF)
- peak allocated VRAM

For true chunked live streaming, use the separate **Qwen3_TTS_Live_vLLM_Colab.ipynb** notebook.


In [ ]:
# 1) GPU check
!nvidia-smi
import torch, platform
print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 2) Install Qwen3-TTS
!pip -q install -U qwen-tts soundfile

# Optional: A100/H100 users can uncomment this for FlashAttention 2.
# It can take a few minutes to compile/install.
# !MAX_JOBS=4 pip -q install -U flash-attn --no-build-isolation


In [ ]:
# 3) Load model
import importlib.util, time, torch, soundfile as sf
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio, display

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"

attn_impl = "flash_attention_2" if importlib.util.find_spec("flash_attn") else "sdpa"
print("Attention implementation:", attn_impl)

t0 = time.perf_counter()
model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation=attn_impl,
)
torch.cuda.synchronize()
print(f"Loaded in {time.perf_counter() - t0:.2f}s")
print("Speakers:", model.get_supported_speakers())
print("Languages:", model.get_supported_languages())


In [ ]:
# 4) Single Japanese TTS test
TEXT = "今日は一緒に日本語を練習しましょう。まず、好きな食べ物について教えてください。"
SPEAKER = "Ono_Anna"
INSTRUCT = "自然な日常会話。親しみやすく、少し明るい声で話してください。"
OUT = "/content/qwen_japanese.wav"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
t0 = time.perf_counter()

wavs, sr = model.generate_custom_voice(
    text=TEXT,
    language="Japanese",
    speaker=SPEAKER,
    instruct=INSTRUCT,
)

torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
wav = wavs[0]
duration = len(wav) / sr
rtf = elapsed / duration
peak_vram = torch.cuda.max_memory_allocated() / 1024**3

sf.write(OUT, wav, sr)

print(f"Generation time : {elapsed:.3f}s")
print(f"Audio duration  : {duration:.3f}s")
print(f"RTF             : {rtf:.3f}  (lower is better; <1 = faster than realtime)")
print(f"Peak VRAM alloc : {peak_vram:.2f} GB")
print("Saved:", OUT)
display(Audio(OUT))


In [ ]:
# 5) Reusable function
def qwen_tts(text, speaker="Ono_Anna", instruct="", output="/content/qwen_output.wav"):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.perf_counter()

    wavs, sr = model.generate_custom_voice(
        text=text,
        language="Japanese",
        speaker=speaker,
        instruct=instruct or None,
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    wav = wavs[0]
    duration = len(wav) / sr
    sf.write(output, wav, sr)

    stats = {
        "generation_s": elapsed,
        "audio_s": duration,
        "rtf": elapsed / duration,
        "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024**3,
        "output": output,
    }
    print(stats)
    display(Audio(output))
    return stats

qwen_tts(
    "えっ、本当ですか？それはちょっとびっくりしました。でも、面白そうですね。",
    instruct="驚きからすぐに親しみのある笑顔に変わる、自然な会話調。"
)


In [ ]:
# 6) Small latency benchmark
TESTS = [
    "こんにちは！今日は何を勉強したいですか？",
    "そうですね。まずこの言葉を声に出して読んでみましょう。",
    "惜しいです。もう一度、ゆっくり発音してみてください。",
]

rows = []
for i, text in enumerate(TESTS, 1):
    out = f"/content/qwen_bench_{i}.wav"
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    wavs, sr = model.generate_custom_voice(
        text=text,
        language="Japanese",
        speaker="Ono_Anna",
        instruct="自然な日本語の会話。"
    )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    dur = len(wavs[0]) / sr
    sf.write(out, wavs[0], sr)
    rows.append((i, elapsed, dur, elapsed/dur, torch.cuda.max_memory_allocated()/1024**3))

print("id | gen_s | audio_s | RTF | peak_VRAM_GB")
for r in rows:
    print(f"{r[0]:>2} | {r[1]:>5.2f} | {r[2]:>7.2f} | {r[3]:>4.2f} | {r[4]:>6.2f}")
